https://aistudio.google.com/

In [3]:
import os
import requests
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수에서 API 키 가져오기
API_KEY = os.getenv("MYREALTRIP_API_KEY")

# API 요청 설정
url = "https://partner-ext-api.myrealtrip.com/v1/revenues"
headers = {
    "Authorization": f"Bearer {API_KEY}"
}
params = {
    "startDate": "2025-01-01",
    "endDate": "2025-01-07",
    "dateSearchType": "SETTLEMENT"
}

# 요청 보내기
response = requests.get(url, headers=headers, params=params)

# 결과 확인
if response.status_code == 200:
    print("성공적으로 데이터를 가져왔습니다!")
    print(response.json())
else:
    print(f"에러 발생: {response.status_code}")
    print(response.text)

성공적으로 데이터를 가져왔습니다!
{'data': [], 'meta': {'totalCount': 0}, 'result': {'status': 200, 'message': 'SUCCESS', 'code': 'success'}}


In [2]:
!pip install google-generativeai

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.30.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_api_python_client-2.194.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.49.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached proto_plus-1.27.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.78.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached grpcio_status-1.76.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.75.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.75.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.74.0-py3-

In [1]:
import os
import requests
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()

# API 키 설정
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-1.5-flash')

def get_marit_products(query):
    # 실제 마리트 API의 상품 검색 엔드포인트와 파라미터를 사용하세요
    url = "https://partner-ext-api.myrealtrip.com/v1/products" # 예시 URL
    headers = {"Authorization": f"Bearer {os.getenv('MYREALTRIP_API_KEY')}"}
    params = {"keyword": query} 
    
    response = requests.get(url, headers=headers, params=params)
    return response.json() if response.status_code == 200 else None

def recommend_trip(user_input):
    # 1. 마리트에서 원본 데이터 가져오기
    raw_data = get_marit_products(user_input)
    
    # 2. Gemini에게 전달할 프롬프트 구성
    prompt = f"""
    사용자 요청: {user_input}
    상품 리스트: {raw_data}
    
    위 상품 리스트 중에서 사용자의 요청에 가장 적합한 상품 3개를 골라줘.
    각 상품별로 추천 이유를 친절하게 설명해주고, 가격과 링크 정보를 포함해줘.
    한국어로 응답해줘.
    """
    
    # 3. Gemini 응답 생성
    response = model.generate_content(prompt)
    return response.text

c:\Users\agnes\용산5기\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\agnes\AppData\Local\Temp\ipykernel_27624\1038083844.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
import streamlit as st
import os
import requests
import google.generativeai as genai
from dotenv import load_dotenv

# 설정 로드
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-1.5-flash')

st.set_page_config(page_title="멍냥여행추천", layout="wide")

# 1. 마이리얼트립 API에서 반려동물 상품 가져오기
def get_pet_tours():
    url = "https://partner-ext-api.myrealtrip.com/v1/products"
    headers = {"Authorization": f"Bearer {os.getenv('MYREALTRIP_API_KEY')}"}
    # '반려동물' 키워드로 검색 (문서의 검색 파라미터 기준)
    params = {"keyword": "반려동물", "limit": 10} 
    
    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        return response.json().get('data', []) # 실제 데이터 구조에 맞춰 조정 필요
    return []

# 2. Gemini를 이용한 테마 추천 로직
def get_ai_recommendation(products):
    prompt = f"""
    아래는 마이리얼트립의 반려동물 동반 여행 상품 리스트야:
    {products}
    
    이 중에서 현재 가장 'Hot'하고 매력적인 테마 3가지를 선정해서 요약해줘.
    형식:
    ### 🐾 [테마 이름]
    - **추천 이유:** 왜 이 테마가 인기 있는지 설명
    - **대표 상품:** 상품명 (가격)
    """
    response = model.generate_content(prompt)
    return response.text

# --- 화면 UI ---
st.title("🐾 반려동물과 함께하는 HOT 테마 여행")
st.subheader("마이리얼트립 실시간 데이터를 AI가 분석한 추천 리스트입니다.")

if st.button('오늘의 추천 테마 확인하기'):
    with st.spinner('AI가 실시간 상품을 분석 중입니다...'):
        # 데이터 호출
        raw_products = get_pet_tours()
        
        if raw_products:
            # AI 분석 결과 출력
            ai_result = get_ai_recommendation(raw_products)
            st.markdown(ai_result)
        else:
            st.error("데이터를 불러오지 못했습니다. API 권한이나 키를 확인해주세요.")

# 하단에 실제 상품 리스트도 살짝 보여주기
st.divider()
st.write("📍 실시간 검색 결과")
# raw_products를 활용해 이미지와 함께 리스트 노출 가능

C:\Users\agnes\AppData\Local\Temp\ipykernel_16308\2294625600.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
2026-04-14 09:43:43.203 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-14 09:43:43.205 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-14 09:43:43.413 
  command:

    streamlit run C:\Users\agnes\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-14 09:43:43.415 Thread 'MainThread': miss

In [ ]:
import os
import requests
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()
API_KEY = os.getenv("MYREALTRIP_API_KEY")

# 1. API 주소 및 헤더 설정
url = "https://partner-ext-api.myrealtrip.com/v1/products/tna/categories"
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"  # POST 요청 시 필수!
}

# 2. 보낼 데이터 (Body) - 서울, 제주 등으로 바꿔보세요
payload = {
    "city": "제주" 
}

# 3. POST 방식으로 API 호출
print(f"'{payload['city']}'의 카테고리를 조회합니다...")
response = requests.post(url, headers=headers, json=payload)

# 4. 결과 확인
if response.status_code == 200:
    print("✅ 조회 성공! (아래 카테고리 목록을 확인하세요)")
    data = response.json()
    
    # 예쁘게 출력해보기
    categories = data.get('categories', [])
    for idx, cat in enumerate(categories, 1):
        print(f"{idx}. {cat.get('name')} (검색용 값: {cat.get('value')})")
        
else:
    print(f"🚨 에러 발생 (상태 코드: {response.status_code})")
    print(response.text)